# Fabric Data Agent MCP - Service Principal (app-only) auth sample

Based on the Microsoft Learn doc ["Data agent as Model Context Protocol server"](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-mcp-server#connect-from-python).

The doc's tip says: *"To run unattended, such as in a service or a job, use a service principal credential instead, for example `ClientSecretCredential` or `DefaultAzureCredential`. The rest of the code stays the same."*

This notebook swaps in `ClientSecretCredential` (client id + client secret + tenant id) so the token represents the **service principal's app-only identity**, not a signed-in user. Everything else - the MCP URL, the handshake, tool discovery, and the call - is unchanged from the user-auth sample.

In [ ]:
%pip install mcp azure-identity --quiet

In [ ]:
import asyncio

from azure.identity import ClientSecretCredential
from mcp import ClientSession

# Defensive import: different mcp package versions/Fabric runtimes expose
# this client under different names.
try:
    from mcp.client.streamable_http import streamablehttp_client
except ImportError:
    from mcp.client.streamable_http import streamable_http_client as streamablehttp_client

tenant_id = "<AAD_TENANT_ID>"
client_id = "<AAD_CLIENT_ID>"          # service principal (app) id
client_secret = "<AAD_CLIENT_SECRET>"  # fill in manually, or pull from Key Vault

workspace_id = "<your-workspace-id>"
data_agent_id = "<your-data-agent-id>"
question = "<your question>"

mcp_url = (
    f"https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}"
    f"/dataagents/{data_agent_id}/agent"
)

In [ ]:
# ClientSecretCredential authenticates as the SERVICE PRINCIPAL itself
# (app-only / client-credentials flow) - no signed-in user is involved.
# The SP must have read access to the workspace and the data agent for this
# token to be accepted by the MCP endpoint.
credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret,
)


def get_auth_headers():
    token = credential.get_token("https://api.fabric.microsoft.com/.default")
    return {"Authorization": f"Bearer {token.token}"}

In [ ]:
async def query_data_agent(question):
    headers = get_auth_headers()

    # This Fabric runtime's streamable_http_client() doesn't accept a
    # 'headers' kwarg directly, so inject the Authorization header via a
    # pre-configured httpx client instead (passed as 'http_client').
    from mcp.shared._httpx_utils import create_mcp_http_client

    http_client = create_mcp_http_client(headers=headers)

    async with streamablehttp_client(mcp_url, http_client=http_client) as streams:
        # This mcp build yields (read, write) - not the usual 3-tuple.
        read, write = streams[0], streams[1]
        async with ClientSession(read, write) as session:
            await session.initialize()

            # The data agent exposes a single tool. Discover it, then call it.
            tools = await session.list_tools()
            tool = tools.tools[0]

            # Different mcp versions name this field inputSchema (camelCase)
            # or input_schema (snake_case) - support both.
            schema = getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None)
            question_arg = next(iter(schema["properties"]))

            result = await session.call_tool(tool.name, {question_arg: question})

            answers = [block.text for block in result.content if block.type == "text"]
            return "\n".join(answers)

In [ ]:
# Run this cell as-is if you're in a Fabric/Jupyter notebook kernel
# (which already runs its own event loop) - top-level `await` works there.
# If running this as a standalone .py script instead, replace the line
# below with: answer = asyncio.run(query_data_agent(question))
answer = await query_data_agent(question)
print(answer)